# BioPred Phase 2 -- Baseline Modeling, Class Balance, and Scaffold-Aware CV

## Purpose

This notebook establishes the first modeling baseline for BioPred using the locked training artifacts created in Notebook 07.

Notebook 07 defined the final scaffold-aware training/test split and saved the raw modeling inputs. Notebook 08 uses only the training set. The final scaffold-held-out test set is not used here.

The goal is to build a simple, defensible modeling baseline before moving into larger model benchmarking, feature ablation, descriptor pruning, or final tuning.

## Inputs

This notebook uses the split-aware artifacts saved by Notebook 07, including:

* raw training features
* primary training labels
* sensitivity labels
* training metadata
* scaffold assignments
* feature QA flags
* feature schema
* preprocessing policy

## Outputs

This notebook will save:

* baseline cross-validation results
* class-balance summaries
* baseline model comparison tables
* notes on which baseline strategies should move forward
* any reusable baseline configuration needed by later modeling notebooks

## Objectives

1. Load the locked training artifacts from Notebook 07.
2. Confirm that the training features, labels, metadata, scaffolds, and QA flags are aligned.
3. Review class balance for the primary label and sensitivity labels using the training set only.
4. Build minimal preprocessing pipelines that handle RDKit missing values inside each cross-validation fold.
5. Set up scaffold-aware cross-validation using the training scaffolds.
6. Train a no-signal dummy baseline and a regularized Logistic Regression baseline on the full frozen feature set.
7. Evaluate baseline models and class-balance strategies using training-set scaffold-aware CV metrics, including PR AUC, ROC AUC, Precision@K%, and EF@K%.
8. Save baseline CV results and identify which approaches are worth carrying forward.



In [21]:
# list imports needed for this notebook
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

# Metrics
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
)


warnings.filterwarnings("ignore")

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

/home/ryanm/projects/BioPred


In [2]:
# define paths for our training artifacts from Notebook 07.
X_TRAIN_PATH = paths.MODELING_DIR / "X_train_raw.parquet"
Y_TRAIN_PATH = paths.MODELING_DIR / "y_train.parquet"
Y_SENSITIVITY_PATH = paths.MODELING_DIR / "y_sensitivity_train.parquet"
METADATA_TRAIN_PATH = paths.MODELING_DIR / "metadata_train.parquet"
FEATURE_QA_FLAGS_TRAIN_PATH = paths.MODELING_DIR / "feature_qa_flags_train.parquet"
SPLIT_ASSIGNMENTS_PATH = paths.MODELING_DIR / "split_assignments_train.parquet"

FEATURE_SCHEMA_PATH = paths.MODELING_DIR / "feature_schema.json"
PREPROCESSING_POLICY_PATH = paths.MODELING_DIR / "preprocessing_policy.json"

In [3]:
# load our training artifacts
X_train_raw = pd.read_parquet(X_TRAIN_PATH)
y_train = pd.read_parquet(Y_TRAIN_PATH)
y_sensitivity_train = pd.read_parquet(Y_SENSITIVITY_PATH)
metadata_train = pd.read_parquet(METADATA_TRAIN_PATH)
feature_qa_flags_train = pd.read_parquet(FEATURE_QA_FLAGS_TRAIN_PATH)
split_assignments_train = pd.read_parquet(SPLIT_ASSIGNMENTS_PATH)

with open(FEATURE_SCHEMA_PATH, "r") as f:
    feature_schema = json.load(f)

with open(PREPROCESSING_POLICY_PATH, "r") as f:
    preprocessing_policy = json.load(f)

In [4]:
# do a verification check on our files before continuing
artifact_shapes = pd.DataFrame(
    {
        "artifact" : [
            "X_train_raw",
            "y_train",
            "y_sensitivity_train",
            "metadata_train",
            "feature_qa_flags_train",
            "split_assignments_train"
        ],
        "rows" : [
            X_train_raw.shape[0],
            y_train.shape[0],
            y_sensitivity_train.shape[0],
            metadata_train.shape[0],
            feature_qa_flags_train.shape[0],
            split_assignments_train.shape[0],
        ],
        "columns" : [
            X_train_raw.shape[1],
            y_train.shape[1],
            y_sensitivity_train.shape[1],
            metadata_train.shape[1],
            feature_qa_flags_train.shape[1],
            split_assignments_train.shape[1],
        ],
    }
)

artifact_shapes

,artifact,rows,columns
0,X_train_raw,1236,2265
1,y_train,1236,1
2,y_sensitivity_train,1236,2
3,metadata_train,1236,12
4,feature_qa_flags_train,1236,1
5,split_assignments_train,1236,3


In [5]:
# confirm all artifacts share the same alignment
training_artifacts = {
    "X_train_raw" : X_train_raw,
    "y_train" : y_train,
    "y_sensitivity_train" : y_sensitivity_train,
    "metadata_train" : metadata_train,
    "feature_qa_flags_train" : feature_qa_flags_train,
    "split_assignments_train" : split_assignments_train,
}

reference_index = X_train_raw.index

alignment_checks = []

for name, artifact in training_artifacts.items():
    alignment_checks.append(
        {
            "artifact" : name,
            "same_index_as_X_train_raw" : artifact.index.equals(reference_index),
            "n_rows" : len(artifact),
            "n_unique_index_values" : artifact.index.nunique(),
            "has_duplicate_index" : artifact.index.has_duplicates,
        }
    )

alignment_checks = pd.DataFrame(alignment_checks)

alignment_checks

,artifact,same_index_as_X_train_raw,n_rows,n_unique_index_values,has_duplicate_index
0,X_train_raw,True,1236,1236,False
1,y_train,True,1236,1236,False
2,y_sensitivity_train,True,1236,1236,False
3,metadata_train,True,1236,1236,False
4,feature_qa_flags_train,True,1236,1236,False
5,split_assignments_train,True,1236,1236,False


In [6]:
# create the target and scaffold contract before we begin the first model
PRIMARY_TARGET = "active_median_px_6_0"

SENSITIVITY_TARGETS = [
    "active_median_px_5_5",
    "active_median_px_6_5"
]

SCAFFOLD_COL = "bemis_murcko_scaffold"

assert list(y_train.columns) == [PRIMARY_TARGET]
assert list(y_sensitivity_train.columns) == SENSITIVITY_TARGETS
assert SCAFFOLD_COL in split_assignments_train.columns

print("Target and scaffold targets confirmed.")
print(f"Primary target: {PRIMARY_TARGET}")
print(f"Sensitivity targets: {SENSITIVITY_TARGETS}")
print(f"Scaffold column: {SCAFFOLD_COL}")

Target and scaffold targets confirmed.
Primary target: active_median_px_6_0
Sensitivity targets: ['active_median_px_5_5', 'active_median_px_6_5']
Scaffold column: bemis_murcko_scaffold


In [7]:
# create our modeling inputs
X = X_train_raw.copy()
y = y_train[PRIMARY_TARGET].copy()
groups = split_assignments_train[SCAFFOLD_COL].copy()

# make a summary of our inputs before we continue
modeling_inputs_summary = {
    "n_samples" : X.shape[0],
    "n_features" : X.shape[1],
    "n_positive" : int(y.sum()),
    "n_negative" : int((1 - y).sum()),
    "positive_rate" : y.mean(),
    "n_scaffolds" : groups.nunique(),
    "largest_scaffold_size" : groups.value_counts().max(),
    "median_scaffold_size" : groups.value_counts().median(),
    "missing_scaffold_values" : groups.isna().sum(),
}

modeling_inputs_summary

{'n_samples': 1236,
 'n_features': 2265,
 'n_positive': 959,
 'n_negative': 277,
 'positive_rate': np.float64(0.7758899676375405),
 'n_scaffolds': 440,
 'largest_scaffold_size': np.int64(59),
 'median_scaffold_size': np.float64(1.0),
 'missing_scaffold_values': np.int64(0)}

The training-side modeling inputs are now defined for Notebook 08.

The feature matrix contains 1,236 training molecules and 2,265 frozen features from Notebook 07. The primary modeling label contains 959 active molecules and 277 inactive molecules, corresponding to an active rate of approximately 77.6%. Scaffold groups are available for scaffold-aware cross-validation using the Bemis-Murcko scaffold assignments, with 440 unique training scaffolds and no missing scaffold values.

These checks confirm that Notebook 08 can proceed using the locked training artifacts only. The final test set remains unused and reserved for later one-time scaffold-held-out evaluation.


### **Section 3: Training-Set Class Balance Review**

This section summarizes the active/inactive counts for the primary label and sensitivity labels using the training set only.

These counts provide the baseline prevalence needed to interpret PR AUC, Precision@K%, EF@K%, and any class-balance strategies evaluated later in scaffold-aware cross-validation.


In [8]:
# combine primary and sensitivity labels for a training-only balance review.
# we did this in Notebook 04, but on the global scale.  This time it will be a training view.
training_labels = pd.concat([
    y_train,
    y_sensitivity_train,
    ], axis = 1
)

label_balance = (
    training_labels
    .agg(["sum", "mean"])
    .T
    .rename(columns = {"sum" : "n_active", "mean" : "active_rate"})
)

label_balance["n_total"] = len(training_labels)

label_balance["n_inactive"] = label_balance["n_total"] - label_balance["n_active"]

label_balance = label_balance[
    ["n_total", "n_active", "n_inactive", "active_rate"]
]

label_balance

,n_total,n_active,n_inactive,active_rate
active_median_px_6_0,1236,959.0,277.0,0.77589
active_median_px_5_5,1236,1044.0,192.0,0.84466
active_median_px_6_5,1236,831.0,405.0,0.67233


The training split preserves the expected label pattern from the full molecule-level dataset: active prevalence decreases as the activity threshold becomes stricter.

The primary label remains active-heavy, with approximately 77.6% active molecules. Even the stricter sensitivity label remains active-majority. Because of this, downstream baseline metrics should be interpreted against high training-set prevalence rather than against a balanced 50/50 classification setting.

This supports using PR AUC, Precision@K%, and EF@K% as key scaffold-aware CV metrics, while treating accuracy-style metrics as secondary diagnostics.


### **Section 4: Fold-Safe Preprocessing Setup**

This section defines the preprocessing structure used inside scaffold-aware cross-validation.

Notebook 07 identified RDKit descriptor missingness and carried the training-side QA flags into `feature_qa_flags_train`. In the locked training split, two molecules have RDKit descriptor missingness flags. RDKit descriptors therefore require imputation inside each cross-validation training fold.

Morgan fingerprint features are binary indicators and will pass through unchanged for the baseline models.

The goal is to build a minimal preprocessing pipeline that is fit only on each fold-training subset and then applied to that fold’s validation subset, avoiding preprocessing leakage.



In [9]:
# print the feature_schema keys for reference
print(feature_schema.keys())

# extract feature groups from the feature schema
rdkit_cols = feature_schema["rdkit_descriptors"]["columns"]
morgan_cols = feature_schema["morgan_fingerprints"]["columns"]

feature_group_summary = {
    "n_total_features_schema" : feature_schema["n_total_features"],
    "n_rdkit_descriptors" : len(rdkit_cols),
    "n_morgan_fingerprints" : len(morgan_cols),
    "n_total_grouped_features" : len(rdkit_cols) + len(morgan_cols),
    "n_X_features" : X.shape[1],
}

feature_group_summary



dict_keys(['n_total_features', 'feature_order', 'rdkit_descriptors', 'morgan_fingerprints', 'primary_target', 'sensitivity_targets'])


{'n_total_features_schema': 2265,
 'n_rdkit_descriptors': 217,
 'n_morgan_fingerprints': 2048,
 'n_total_grouped_features': 2265,
 'n_X_features': 2265}

In [10]:
# confirm schema-defined feature groups match the training feature matrix.
all_schema_cols = rdkit_cols + morgan_cols

schema_feature_checks = {
    "schema_total_matches_X" : feature_schema["n_total_features"] == X.shape[1],
    "grouped_total_matches_schema" : feature_group_summary["n_total_grouped_features"] == X.shape[1],
    "all_schema_cols_in_X" : set(all_schema_cols).issubset(set(X.columns)),
    "all_X_cols_in_schema" : set(X.columns).issubset(set(all_schema_cols)),
    "n_duplicate_schema_cols" : len(X.columns) - len(set(all_schema_cols)),
}

schema_feature_checks

{'schema_total_matches_X': True,
 'grouped_total_matches_schema': True,
 'all_schema_cols_in_X': True,
 'all_X_cols_in_schema': True,
 'n_duplicate_schema_cols': 0}

The saved feature schema matches the locked training feature matrix.

The RDKit descriptor and Morgan fingerprint column groups account for all 2,265 training features, with no missing or duplicated schema columns. This allows Notebook 08 to use the schema to route RDKit descriptors and Morgan fingerprints through separate preprocessing branches without relying on fragile column-name inference.


In [11]:
# build our preprocessing pipeline
# RDKit descriptors receive median imputation and scaling.
rdkit_preprocessor = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "median")),
        ("scaler", StandardScaler()),
    ]
)


# make the ColumnTransformer for the rdkit and morgan cols
preprocessor = ColumnTransformer(
    transformers = [
        ("rdkit", rdkit_preprocessor, rdkit_cols),
        ("morgan", "passthrough", morgan_cols),
    ],
    remainder = "drop"
)


The preprocessing object routes feature groups according to the saved Notebook 07 schema.

RDKit descriptors receive median imputation and scaling because they are continuous-valued descriptors with limited missingness. Morgan fingerprints pass through unchanged because they are binary structural indicators. This preprocessor is defined here but will be fit only inside scaffold-aware cross-validation folds.


In [12]:
 # before we fit lets just create a quick audit table before we enter the CV setup

preprocessing_summary = pd.DataFrame(
    [
        {
            "branch" : "rdkit",
            "n_features" : len(rdkit_cols),
            "preprocessing" : "preprocessing imputation + standard scaling.",
            "fit_timing" : "inside CV fold only.",
        },
        {
            "branch" : "morgan",
            "n_features" : len(morgan_cols),
            "preprocessing" : "passthrough",
            "fit_timing" : "not fit; passed through inside the CV fold.",
        },
    ]
)

preprocessing_summary

,branch,n_features,preprocessing,fit_timing
0,rdkit,217,preprocessing imputation + standard scaling.,inside CV fold only.
1,morgan,2048,passthrough,not fit; passed through inside the CV fold.


The preprocessing setup now separates the frozen feature matrix into two schema-defined branches.

RDKit descriptors will receive fold-internal median imputation and scaling. Morgan fingerprints will pass through unchanged. The preprocessing object has been defined but not fit globally, preserving the training-fold-only preprocessing boundary required for scaffold-aware cross-validation.


### **Section 5: Scaffold-Aware Cross-Validation Setup**

This section defines the cross-validation strategy used for baseline modeling.

`StratifiedGroupKFold` will split the locked training set while using the primary label for stratification and Bemis-Murcko scaffolds as grouping constraints. This keeps molecules with the same scaffold together in the same fold while trying to preserve active/inactive balance across folds.

The goal is to evaluate baseline models on scaffold-separated validation folds without using the final held-out test set.


In [13]:
# define scaffold-aware cross-validation splitter.
# Stratification uses the primary label; grouping uses the Bemis-Murcko scaffolds.
# use the same seed as the Notebook 07 scaffold split for traceability.

SELECTED_SPLIT_SEED = 33

cv = StratifiedGroupKFold(
    n_splits = 5,
    shuffle = True,
    random_state = SELECTED_SPLIT_SEED
)

In [14]:
# audit the scaffold-cv folds, making sure we produced reasonable validation folds

cv_fold_rows = []

for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups = groups), start = 1):
    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    train_groups = groups.iloc[train_idx]
    valid_groups = groups.iloc[valid_idx]

    scaffold_overlap = set(valid_groups).intersection(set(train_groups))

    cv_fold_rows.append(
        {
            "fold" : fold_id,
            "n_train" : len(train_idx),
            "n_valid" : len(valid_idx),
            "valid_n_active" : int(y_valid_fold.sum()),
            "valid_n_inactive" : int((y_valid_fold == 0).sum()),
            "valid_active_rate" : float(y_valid_fold.mean()),
            "train_n_scaffolds" : train_groups.nunique(),
            "valid_n_scaffolds" : valid_groups.nunique(),
            "n_scaffold_overlap" : len(scaffold_overlap)
        }
    )

cv_fold_audit = pd.DataFrame(cv_fold_rows)

assert (cv_fold_audit["n_scaffold_overlap"] == 0).all()
assert cv_fold_audit["valid_n_active"].min() > 0
assert cv_fold_audit["valid_n_inactive"].min() > 0

print("Scaffold-CV fold audit passed.")

cv_fold_audit

Scaffold-CV fold audit passed.


,fold,n_train,n_valid,valid_n_active,valid_n_inactive,valid_active_rate,train_n_scaffolds,valid_n_scaffolds,n_scaffold_overlap
0,1,1047,189,149,40,0.788360,348,92,0
1,2,1001,235,194,41,0.825532,354,86,0
2,3,1004,232,201,31,0.866379,356,84,0
3,4,1028,208,149,59,0.716346,353,87,0
4,5,864,372,266,106,0.715054,349,91,0


The scaffold-aware CV audit confirms that validation folds contain no scaffold overlap with their corresponding training folds.

Fold sizes and active rates are not perfectly balanced, which is expected when stratification is constrained by scaffold grouping. Because fold composition varies, baseline results should be summarized with both mean and standard deviation across folds rather than relying on a single aggregate score.


### **Section 6: Baseline Model Definitions**

This section defines the initial baseline models for scaffold-aware cross-validation.

The first baseline is a no-signal `DummyClassifier`, which provides a sanity-check floor for ranking metrics under the observed training-label prevalence. The second baseline family is regularized Logistic Regression, evaluated with both unweighted and class-balanced loss settings.

These models are intentionally simple. The goal is to test whether the frozen structure-derived features contain scaffold-generalizable ranking signal before introducing heavier model families or feature-selection experiments.


In [15]:
# define the baseline estimators

# no-signal baseline: predicts the majority class observed in each fold-training split.
dummy_model = DummyClassifier(
    strategy = "most_frequent"
)

# regularized Logistic Regression baseline with equal sample weighting.
logreg_unweighted = LogisticRegression(
    max_iter = 5000,
    solver = "liblinear",
    class_weight = None,
    random_state = SELECTED_SPLIT_SEED
)

# regularized Logistic Regression baseline with class-balanced loss weighting.
logreg_balanced = LogisticRegression(
    max_iter = 5000,
    solver = "liblinear",
    class_weight = "balanced",
    random_state = SELECTED_SPLIT_SEED,
)

The baseline estimators are now defined but have not been fit globally.

This preserves the scaffold-aware CV boundary: each estimator will be fit only inside the fold-level evaluation loop, together with the preprocessing pipeline.


### **Section 7: Baseline Evaluation with Scaffold-Aware CV Metrics**

This section combines the preprocessing pipeline, baseline estimators, and scaffold-aware CV splitter into a fold-level evaluation workflow.

For each validation fold, the preprocessing steps and model are fit only on that fold’s training subset. Predictions are then generated for the scaffold-separated validation subset and evaluated with ranking-oriented metrics.

The evaluation focuses on PR AUC, ROC AUC, Precision@K%, and EF@K%. Balanced accuracy is included only as a secondary threshold diagnostic.


In [17]:
# we need to first define our metrics
# both Prec@K and EF@K% are not built into sklearn, so we need to compute manually.

def precision_at_k_percent(y_true, y_score, k_percent):
    """
    Compute Precision@K% for a ranked validation fold.

    Parameters
    ----------
    y_true : Binary validation labels where 1 indicates active and 0 indicates inactive.

    y_score : Model scores used for ranking validation molecules. Higher scores should
        indicate higher predicted probability of activity.

    k_percent : Fraction of validation molecules to include in the top-ranked subset.
        Use decimal form, e.g. 0.05 for top 5% and 0.10 for top 10%.

    Returns
    -------
    float
        Fraction of active molecules among the top K% highest-scored validation
        molecules. The selected K is computed as ceil(n_samples * k_percent),
        with a minimum of 1 molecule.

    """
    y_true = pd.Series(y_true).reset_index(drop = True)
    y_score = pd.Series(y_score).reset_index(drop = True)

    k = max(1, int(np.ceil(len(y_true) * k_percent)))

    top_k_idx = y_score.sort_values(ascending = False).index[:k]

    return float(y_true.iloc[top_k_idx].mean())



In [19]:
# and then enrichment_factor function
# requires previous precision@k function
def enrichment_factor_at_k_percent(y_true, y_score, k_percent):
    """
    Compute EF@K% for a ranked validation fold.

    EF@K% compares the active fraction in the top K% highest-scored molecules
    against the active fraction in the full validation fold.

    Parameters
    ----------
    y_true : Binary validation labels where 1 indicates active and 0 indicates inactive.

    y_score : Model scores used for ranking validation molecules. Higher scores should
        indicate higher predicted probability of activity.

    k_percent : Fraction of validation molecules to include in the top-ranked subset.
        Use decimal form, e.g. 0.05 for top 5% and 0.10 for top 10%.

    Returns
    -------
    float
        Enrichment factor among the top K% highest-scored validation molecules.
        Values above 1.0 indicate enrichment over random selection from the same
        validation fold. Returns np.nan if the validation fold contains no actives.    
    """
    y_true = pd.Series(y_true).reset_index(drop = True)
    
    baseline_active_rate = y_true.mean()

    precision_k = precision_at_k_percent(
        y_true = y_true,
        y_score = y_score,
        k_percent = k_percent
    )

    return float(precision_k / baseline_active_rate)

In [ ]:
# test the functions before we employ them
base_y_true = pd.Series([1, 0, 1, 1, 0])
base_y_score = pd.Series([0.92, 0.81, 0.76, 0.41, 0.20])

base_metric_check = {
    "precision_at_40_percent" : precision_at_k_percent(
        y_true = base_y_true,
        y_score = base_y_score,
        k_percent = 0.40,
    ),
    "ef_at_40_percent" : enrichment_factor_at_k_percent(
        y_true = base_y_true,
        y_score = base_y_score,
        k_percent = 0.40,
    ),
    "baseline_active_rate" : base_y_true.mean()
}

base_metric_check

# Expected output should be approximately:
# Top 40% of 5 molecules = top 2 molecules
# Top 2 labels = [1,0]
# Precision@40% = 0.50
# Baseline active rate = 3/5 = 0.60
# EF@40% = (0.50 / 0.60) = 0.83333

{'precision_at_40_percent': 0.5,
 'ef_at_40_percent': 0.8333333333333334,
 'baseline_active_rate': np.float64(0.6)}

In [22]:
# create and compute function for our baseline models to run, we will combine all of our parts we have run through in this notebook together.
# we will test the function on the DummyClassifier first

def evaluate_baseline_model_with_scaffold_cv(model, model_name, X, y, groups, cv, preprocessor):
    """
    Evaluate one estimator with scaffold-aware CV.

    The preprocessor and model are fit only on each fold-training subset.
    Metrics are computed on the corresponding scaffold-separated validation fold. 
    
    """
    fold_rows = []

    # assemble our parts, the code will look the same/similar to the above cells
    for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups = groups), start = 1):
        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        pipeline = Pipeline(
            steps = [
                ("preprocessor", clone(preprocessor)),
                ("model", clone(model)),
            ]
        )

        pipeline.fit(X_train_fold, y_train_fold)

        y_valid_score = pipeline.predict_proba(X_valid_fold)[:, 1]

        y_valid_pred = pipeline.predict(X_valid_fold)

        fold_rows.append(
            {
                "model_name" : model_name,
                "fold" : fold_id,
                "n_train" : len(train_idx),
                "n_valid" : len(valid_idx),
                "valid_active_rate" : float(y_valid_fold.mean()),
                "average_precision" : average_precision_score(y_valid_fold, y_valid_score),
                "roc_auc" : roc_auc_score(y_valid_fold, y_valid_score),
                "balanced_accuracy" : balanced_accuracy_score(y_valid_fold, y_valid_pred),
                "precision_at_5_percent" : precision_at_k_percent(
                    y_true = y_valid_fold,
                    y_score = y_valid_score,
                    k_percent = 0.05,
                ),
                "ef_at_5_percent" : enrichment_factor_at_k_percent(
                    y_true = y_valid_fold,
                    y_score = y_valid_score,
                    k_percent = 0.05,
                ),
                "precision_at_10_percent" : precision_at_k_percent(
                    y_true = y_valid_fold,
                    y_score = y_valid_score,
                    k_percent = 0.10,
                ),
                "ef_at_10_percent" : enrichment_factor_at_k_percent(
                    y_true = y_valid_fold,
                    y_score = y_valid_score,
                    k_percent = 0.10,
                ),
            }
        )

    return pd.DataFrame(fold_rows)

In [23]:
# test on our dummy classifier
dummy_cv_results = evaluate_baseline_model_with_scaffold_cv(
    model = dummy_model, # created previously above
    model_name = "dummy_most_frequent",
    X = X,
    y = y,
    groups = groups,
    cv = cv,
    preprocessor = preprocessor,
)

dummy_cv_results

,model_name,fold,n_train,n_valid,valid_active_rate,average_precision,roc_auc,balanced_accuracy,precision_at_5_percent,ef_at_5_percent,precision_at_10_percent,ef_at_10_percent
0,dummy_most_frequent,1,1047,189,0.788360,0.788360,0.5,0.5,1.000000,1.268456,0.842105,1.068174
1,dummy_most_frequent,2,1001,235,0.825532,0.825532,0.5,0.5,0.916667,1.110395,0.958333,1.160868
2,dummy_most_frequent,3,1004,232,0.866379,0.866379,0.5,0.5,0.833333,0.961857,0.666667,0.769486
3,dummy_most_frequent,4,1028,208,0.716346,0.716346,0.5,0.5,0.727273,1.015253,0.761905,1.063599
4,dummy_most_frequent,5,864,372,0.715054,0.715054,0.5,0.5,0.894737,1.251286,0.815789,1.140879


The no-signal dummy baseline behaves as expected. Average precision equals the validation-fold active rate, while ROC AUC and balanced accuracy remain at 0.5 across folds.

Top-K metrics for the dummy model are not interpreted as ranking skill because all molecules receive tied scores. Their fold-to-fold variation reflects arbitrary tie ordering rather than learned prioritization. The dummy model is retained only as a sanity-check floor for the scaffold-CV evaluation workflow.


In [ ]:
# evaluate on the unweighted LogReg model
logreg_unweighted_cv_results = evaluate_baseline_model_with_scaffold_cv(
    model = logreg_unweighted,
    model_name = "logreg_unweighted",
    X = X,
    y = y,
    groups = groups,
    cv = cv,
    preprocessor= preprocessor,
)



In [25]:
# now evaluate on our third baseline, the balanced LogReg.
logreg_balanced_cv_results = evaluate_baseline_model_with_scaffold_cv(
    model = logreg_balanced,
    model_name= "logreg_balanced",
    X = X,
    y = y,
    groups = groups,
    cv = cv,
    preprocessor= preprocessor
)


In [26]:
# combine all of our results to view
baseline_cv_results = pd.concat(
    [
        dummy_cv_results,
        logreg_unweighted_cv_results,
        logreg_balanced_cv_results,
    ],
    axis = 0,
    ignore_index= True
)

baseline_cv_results

,model_name,fold,n_train,n_valid,valid_active_rate,average_precision,roc_auc,balanced_accuracy,precision_at_5_percent,ef_at_5_percent,precision_at_10_percent,ef_at_10_percent
0,dummy_most_frequent,1,1047,189,0.788360,0.788360,0.500000,0.500000,1.000000,1.268456,0.842105,1.068174
1,dummy_most_frequent,2,1001,235,0.825532,0.825532,0.500000,0.500000,0.916667,1.110395,0.958333,1.160868
2,dummy_most_frequent,3,1004,232,0.866379,0.866379,0.500000,0.500000,0.833333,0.961857,0.666667,0.769486
3,dummy_most_frequent,4,1028,208,0.716346,0.716346,0.500000,0.500000,0.727273,1.015253,0.761905,1.063599
4,dummy_most_frequent,5,864,372,0.715054,0.715054,0.500000,0.500000,0.894737,1.251286,0.815789,1.140879
5,logreg_unweighted,1,1047,189,0.788360,0.920249,0.848154,0.689010,0.800000,1.014765,0.894737,1.134935
6,logreg_unweighted,2,1001,235,0.825532,0.969047,0.869248,0.786648,1.000000,1.211340,1.000000,1.211340
7,logreg_unweighted,3,1004,232,0.866379,0.968605,0.836944,0.667389,1.000000,1.154229,1.000000,1.154229
8,logreg_unweighted,4,1028,208,0.716346,0.988683,0.971676,0.876408,1.000000,1.395973,1.000000,1.395973
9,logreg_unweighted,5,864,372,0.715054,0.883605,0.770606,0.692793,1.000000,1.398496,0.921053,1.288089


The fold-level baseline results show clear separation between the no-signal dummy baseline and the Logistic Regression baselines.

The dummy model behaves as expected: average precision matches each validation fold’s active rate, while ROC AUC and balanced accuracy remain at 0.5. Its top-K metrics are not interpreted as ranking skill because all molecules receive tied scores.

Both Logistic Regression baselines show substantially stronger scaffold-CV performance. Average precision and ROC AUC are consistently above the dummy reference, indicating that the frozen structure-derived features contain useful ranking signal under scaffold-separated validation. Top-K screening metrics are also strong, with several folds achieving active-only top-ranked subsets at 5% and 10%.

The unweighted and class-balanced Logistic Regression models perform very similarly across ranking metrics. This suggests that class weighting is not a major driver of baseline ranking performance for the primary label, though it may still affect threshold-dependent diagnostics such as balanced accuracy.

These results should be interpreted as promising baseline evidence, not as a discovery claim. The next step is to summarize mean and variance across folds and run targeted sanity checks before drawing stronger conclusions.


In [27]:
# summarize mean/std across all folds
metric_cols = [
    "valid_active_rate",
    "average_precision",
    "roc_auc",
    "balanced_accuracy",
    "precision_at_5_percent",
    "ef_at_5_percent",
    "precision_at_10_percent",
    "ef_at_10_percent",
]

baseline_cv_summary = (
    baseline_cv_results
    .groupby("model_name")[metric_cols]
    .agg(["mean", "std"])
)

baseline_cv_summary

valid_active_rate           average_precision            \
                                 mean       std              mean       std   
model_name                                                                    
dummy_most_frequent          0.782334  0.066796          0.782334  0.066796   
logreg_balanced              0.782334  0.066796          0.946253  0.042205   
logreg_unweighted            0.782334  0.066796          0.946038  0.043070   

                      roc_auc           balanced_accuracy            \
                         mean       std              mean       std   
model_name                                                            
dummy_most_frequent  0.500000  0.000000          0.500000  0.000000   
logreg_balanced      0.858641  0.072279          0.751787  0.078018   
logreg_unweighted    0.859326  0.072832          0.742450  0.087826   

                    precision_at_5_percent           ef_at_5_percent  \
                                      mean       std            mean   
model_name                                                             
dummy_most_frequent               0.874402  0.101622        1.121450   
logreg_balanced                   0.960000  0.089443        1.234961   
logreg_unweighted                 0.960000  0.089443        1.234961   

                              precision_at_10_percent            \
                          std                    mean       std   
model_name                                                        
dummy_most_frequent  0.137239                0.808960  0.107144   
logreg_balanced      0.164492                0.963158  0.051299   
logreg_unweighted    0.164492                0.963158  0.051299   

                    ef_at_10_percent            
                                mean       std  
model_name                                      
dummy_most_frequent         1.040601  0.157569  
logreg_balanced             1.236913  0.107016  
logreg_unweighted           1.236913  0.107016

The scaffold-CV summary shows that both Logistic Regression baselines substantially outperform the no-signal dummy baseline across the main ranking metrics.

The dummy model behaves as expected: its average precision tracks validation-fold prevalence, while ROC AUC and balanced accuracy remain at chance level. In contrast, the Logistic Regression baselines achieve higher mean average precision, higher mean ROC AUC, stronger balanced accuracy, and stronger top-K screening metrics.

The unweighted and class-balanced Logistic Regression models perform nearly identically on ranking-oriented metrics. This suggests that class weighting is not a major driver of baseline ranking performance for the primary label.

Because fold-to-fold variation is visible, these results should be interpreted using both mean and standard deviation rather than a single headline metric. The summary supports a promising baseline signal, pending additional sanity checks.


In [30]:
# compute AP lift over fold prevalence

baseline_cv_results["average_precision_lift_over_prevalence"] = (
    baseline_cv_results["average_precision"]
    / baseline_cv_results["valid_active_rate"]
)

baseline_cv_results[
    [
        "model_name",
        "fold",
        "valid_active_rate",
        "average_precision",
        "average_precision_lift_over_prevalence"
    ]
]

,model_name,fold,valid_active_rate,average_precision,average_precision_lift_over_prevalence
0,dummy_most_frequent,1,0.788360,0.788360,1.000000
1,dummy_most_frequent,2,0.825532,0.825532,1.000000
2,dummy_most_frequent,3,0.866379,0.866379,1.000000
3,dummy_most_frequent,4,0.716346,0.716346,1.000000
4,dummy_most_frequent,5,0.715054,0.715054,1.000000
5,logreg_unweighted,1,0.788360,0.920249,1.167296
6,logreg_unweighted,2,0.825532,0.969047,1.173845
7,logreg_unweighted,3,0.866379,0.968605,1.117992
8,logreg_unweighted,4,0.716346,0.988683,1.380175
9,logreg_unweighted,5,0.715054,0.883605,1.235718


In [31]:
# inspect our top-k counts per fold
top_k_count_summary = cv_fold_audit.assign(
    top_5_percent_n = lambda df : np.ceil(df["n_valid"] * 0.05).astype(int),
    top_10_percent_n = lambda df : np.ceil(df["n_valid"] * 0.10).astype(int),
)

top_k_count_summary[
    [
        "fold",
        "n_valid",
        "top_5_percent_n",
        "top_10_percent_n",
        "valid_active_rate",
    ]
]

,fold,n_valid,top_5_percent_n,top_10_percent_n,valid_active_rate
0,1,189,10,19,0.788360
1,2,235,12,24,0.825532
2,3,232,12,24,0.866379
3,4,208,11,21,0.716346
4,5,372,19,38,0.715054


The AP lift check confirms that the Logistic Regression baselines improve average precision relative to each validation fold’s active-rate baseline. The dummy model has AP lift equal to 1.0 in every fold, as expected, while both Logistic Regression baselines exceed the fold-local prevalence baseline across all folds.

The top-K denominator check shows that Precision@5% and Precision@10% are based on validation shortlists of approximately 10–38 molecules, depending on fold size. These top-K results are therefore small-batch screening metrics, but they are not single-molecule artifacts.

Together, these checks support the interpretation that Logistic Regression is learning useful ranking signal from the frozen structure-derived features under scaffold-aware CV. The results remain baseline evidence rather than discovery claims, and later support/similarity audits should assess whether high-ranked validation molecules are scaffold-disjoint but still close to training chemistry.


In [32]:
# run just one sanity audit for baseline cv results
# we will save what we need after this.

forbidden_feature_cols = [
    PRIMARY_TARGET,
    *SENSITIVITY_TARGETS,
    "median_px",
    "mean_px",
    "min_px",
    "max_px",
    "px_std",
    "px_range"
]

sanity_audit = {
    "n_X_rows" : X.shape[0],
    "n_y_rows" : y.shape[0],
    "n_groups_rows" : groups.shape[0],
    "X_y_index_aligned" : X.index.equals(y.index),
    "X_groups_index_aligned" : X.index.equals(groups.index),
    "n_forbidden_cols_in_X" : len(set(forbidden_feature_cols).intersection(set(X.columns))),
    "all_cv_folds_scaffold_disjoint" : (cv_fold_audit["n_scaffold_overlap"] == 0).all(),
    "final_test_artifacts_used" : False
}

sanity_audit

{'n_X_rows': 1236,
 'n_y_rows': 1236,
 'n_groups_rows': 1236,
 'X_y_index_aligned': True,
 'X_groups_index_aligned': True,
 'n_forbidden_cols_in_X': 0,
 'all_cv_folds_scaffold_disjoint': np.True_,
 'final_test_artifacts_used': False}

The compact sanity audit confirms that the baseline CV evaluation used aligned training features, labels, and scaffold groups; excluded known target and potency-derived forbidden columns from the feature matrix; and preserved scaffold-disjoint validation folds.

The final held-out test artifacts were not used in this notebook. Baseline results therefore remain training-set scaffold-CV estimates rather than final test performance claims.


In [33]:
# save Notebook 08 baseline evaluation artifacts
BASELINE_CV_RESULTS_PATH = paths.MODELING_DIR / "baseline_cv_results.parquet"
BASELINE_CV_SUMMARY_PATH = paths.MODELING_DIR / "baseline_cv_summary.parquet"

baseline_cv_results.to_parquet(BASELINE_CV_RESULTS_PATH)
baseline_cv_summary.to_parquet(BASELINE_CV_SUMMARY_PATH)

saved_baseline_artifacts = {
    "baseline_cv_results" : BASELINE_CV_RESULTS_PATH,
    "baseline_cv_summary" : BASELINE_CV_SUMMARY_PATH,
}

saved_baseline_artifacts

{'baseline_cv_results': PosixPath('/home/ryanm/projects/BioPred/data/processed/modeling/baseline_cv_results.parquet'),
 'baseline_cv_summary': PosixPath('/home/ryanm/projects/BioPred/data/processed/modeling/baseline_cv_summary.parquet')}

In [34]:
# do a reload check to make sure we can recall

baseline_cv_results_reloaded = pd.read_parquet(BASELINE_CV_RESULTS_PATH)
baseline_cv_summary_reloaded = pd.read_parquet(BASELINE_CV_SUMMARY_PATH)

reload_check = {
    "baseline_cv_results_shape_matches": baseline_cv_results_reloaded.shape == baseline_cv_results.shape,
    "baseline_cv_summary_shape_matches": baseline_cv_summary_reloaded.shape == baseline_cv_summary.shape,
}

reload_check

{'baseline_cv_results_shape_matches': True,
 'baseline_cv_summary_shape_matches': True}

The fold-level baseline CV results and grouped summary table were saved as modeling artifacts.

The saved fold-level table preserves per-fold metrics for each baseline model, including validation active rate, average precision, ROC AUC, balanced accuracy, Precision@K%, and EF@K%. The saved summary table preserves mean and standard deviation across scaffold-CV folds for each model.

These artifacts provide the baseline reference point for Notebook 09 candidate model benchmarking.


**Baseline carry-forward decision**

The no-signal dummy model is retained only as a sanity-check reference. It is not a candidate model to carry forward because it provides no learned ranking signal.

Both Logistic Regression baselines substantially outperform the dummy baseline across scaffold-CV ranking metrics. The unweighted and class-balanced versions perform nearly identically on average precision, ROC AUC, Precision@K%, and EF@K%. Because the unweighted Logistic Regression model is simpler and achieves equivalent ranking performance, it is selected as the primary incumbent baseline for Notebook 09.

The class-balanced Logistic Regression result is retained as a documented class-balance comparison, but it is not treated as the preferred baseline unless later threshold-focused analysis specifically favors it.


### **Notebook 08 Summary**

Notebook 08 established the baseline modeling reference for BioPred Phase 1.

Using the locked training artifacts from Notebook 07, the notebook confirmed training-set class balance, defined fold-safe preprocessing, constructed scaffold-aware cross-validation, and evaluated a no-signal dummy baseline alongside unweighted and class-balanced Logistic Regression baselines.

The dummy baseline behaved as expected, with average precision tracking validation-fold prevalence and chance-level ROC AUC and balanced accuracy. Both Logistic Regression baselines substantially outperformed the dummy baseline across scaffold-CV folds, showing strong average precision, ROC AUC, and top-K screening yield.

The unweighted and class-balanced Logistic Regression models performed nearly identically on ranking-oriented metrics. Because the unweighted model is simpler, fast, and operationally lightweight, it should be treated as the incumbent baseline for Notebook 09 candidate benchmarking.

These results are interpreted as promising scaffold-CV baseline evidence, not as final held-out test performance or prospective discovery claims. The protected scaffold-held-out test set remains unused.
